# Day 8: Security - Authentication, Authorization, and Password Management

## Goal
Implement industry-standard JWT authentication with proper password hashing, token refresh, and role-based access control.

## Why This Day Matters
Most data breaches come from weak authentication. Get this right: no usernames in tokens, secure password storage, proper token expiration.

## PART 1: PASSWORD HASHING

Never store plain passwords. Hash them.

## PART 2: JWT TOKENS

Create signed tokens that expire.

## PART 3: LOGIN ENDPOINT

Issue tokens after password verification.

In [ ]:
,
,
token")

# Pseudo-code
example_login = '''
@app.post("/token", response_model=Token)
async def login(
    form_data: OAuth2PasswordRequestForm = Depends(),
    db: Session = Depends(get_db)
):
    """OAuth2 standard login endpoint"""
    
    # Find user
    user = db.query(User).filter(User.username == form_data.username).first()
    
    # Verify password
    if not user or not verify_password(form_data.password, user.password_hash):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid credentials"
        )
    
    # Create token
    access_token = create_access_token(
        data={
            "username": user.username,
            "user_id": user.id,
            "role": user.role
        },
        expires_delta=timedelta(minutes=30)
    )
    
    return {
        "access_token": access_token,
        "token_type": "bearer",
        "expires_in": 30 * 60
    }
'''

print("✓ Login endpoint pattern:")
print(example_login)
print("\n✓ OAuth2PasswordRequestForm provides:")
print("  - username and password fields")
print("  - /docs automatically shows a 'Try it out' button")
]
)

## PART 4: PROTECTED ENDPOINTS

Use tokens to protect routes.

In [ ]:
,
,
    token_data = decode_access_token(token)
    if token_data is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )
    return token_data

@app.get("/me")
def get_current_user_profile(current_user: TokenData = Depends(get_current_user)):
    """Protected endpoint - requires valid token"""
    return {
        "username": current_user.username,
        "user_id": current_user.user_id,
        "role": current_user.role
    }

@app.delete("/my-account")
def delete_account(current_user: TokenData = Depends(get_current_user)):
    """Another protected endpoint"""
    # Delete user with current_user.user_id
    return {"deleted": True}
'''

print("✓ Protected endpoints:")
print(example_protected)
print("\n✓ If user doesn't provide token:")
print("  401 Unauthorized")
print("\n✓ If token invalid or expired:")
print("  401 Unauthorized with 'Invalid token'")
]
)

,
,
def refresh_token(
    request: TokenRefreshRequest,
    db: Session = Depends(get_db)
):
    # Decode refresh token
    token_data = decode_access_token(request.refresh_token)
    if not token_data:
        raise HTTPException(status_code=401, detail="Invalid refresh token")
    
    # Create NEW access token
    new_access_token = create_access_token(
        data={
            "username": token_data.username,
            "user_id": token_data.user_id,
            "role": token_data.role
        },
        expires_delta=timedelta(minutes=15)
    )
    
    return {
        "access_token": new_access_token,
        "token_type": "bearer",
        "expires_in": 15 * 60
    }
'''

print("✓ Refresh token pattern:")
print("  1. User login -> short access token + long refresh token")
print("  2. Access token expires after 15 minutes")
print("  3. App sends refresh token to /token/refresh")
print("  4. Get new access token for next 15 minutes")
print("  5. Refresh token valid for 7 days")
]
)

## PART 7: COMMON SECURITY MISTAKES

### ❌ Mistake 1: Storing plain passwords
Bad: `user.password = password`
Good: `user.password_hash = hash_password(password)`

### ❌ Mistake 2: Weak hashing (MD5, SHA1)
Bad: `hashlib.md5(password).hexdigest()`
Good: `CryptContext(["bcrypt"])`

### ❌ Mistake 3: Storing secret key in code
Bad: `SECRET_KEY = "hardcoded_secret"`
Good: `SECRET_KEY = os.environ["SECRET_KEY"]`

### ❌ Mistake 4: Tokens that don't expire
Bad: No expiration
Good: `"exp": datetime.utcnow() + timedelta(minutes=30)`

### ❌ Mistake 5: User data in token
Bad: `{"username": "alice", "password": "..."}`
Good: `{"user_id": 1, "role": "user"}` (data comes from DB)

### ❌ Mistake 6: No rate limiting on login
Bad: No limit on attempts
Good: Lock after 5 failed attempts